In [1]:
!pip install pandas numpy matplotlib scikit-learn sqlalchemy mysql-connector-python joblib

In [3]:
import os
import re
import getpass
import joblib
import warnings
import pandas as pd
import matplotlib.pyplot as plt

from urllib.parse import quote_plus
from sqlalchemy import create_engine, text, inspect

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

mysql_username = "root"
mysql_host = "localhost"
mysql_port = "3306"
database_name = "government_analytics"
source_table = "bengaluru_civic_complaints"

mysql_password = getpass.getpass("Enter MySQL password: ")
encoded_password = quote_plus(mysql_password)

server_engine = create_engine(
    f"mysql+mysqlconnector://{mysql_username}:{encoded_password}"
    f"@{mysql_host}:{mysql_port}",
    pool_pre_ping=True
)

with server_engine.begin() as connection:
    connection.execute(
        text(
            f"CREATE DATABASE IF NOT EXISTS `{database_name}`"
        )
    )

database_engine = create_engine(
    f"mysql+mysqlconnector://{mysql_username}:{encoded_password}"
    f"@{mysql_host}:{mysql_port}/{database_name}",
    pool_pre_ping=True
)

with database_engine.connect() as connection:
    connected_database = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

print("Connected database:", connected_database)

existing_tables = inspect(database_engine).get_table_names()

if source_table in existing_tables:
    complaints_df = pd.read_sql(
        f"SELECT * FROM `{source_table}`",
        database_engine
    )
    print("Dataset loaded from MySQL.")
else:
    possible_paths = [
        "bengaluru_civic_complaints(1).csv",
        "bengaluru_civic_complaints.csv",
        "/mnt/data/bengaluru_civic_complaints(1).csv",
        r"C:\Users\Kishore\Downloads\bengaluru_civic_complaints(1).csv",
        r"C:\Users\Kishore\Downloads\bengaluru_civic_complaints.csv"
    ]

    csv_path = next(
        (
            path
            for path in possible_paths
            if os.path.exists(path)
        ),
        None
    )

    if csv_path is None:
        raise FileNotFoundError(
            "Place bengaluru_civic_complaints(1).csv "
            "in the same folder as the notebook."
        )

    complaints_df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    complaints_df.to_sql(
        name=source_table,
        con=database_engine,
        if_exists="replace",
        index=False,
        chunksize=1000
    )

    print("Dataset loaded from CSV and uploaded to MySQL.")

if complaints_df.empty:
    raise ValueError("The dataset is empty.")

complaints_df.columns = (
    complaints_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

unnamed_columns = [
    column
    for column in complaints_df.columns
    if column.startswith("unnamed")
]

complaints_df = complaints_df.drop(
    columns=unnamed_columns,
    errors="ignore"
)

title_candidates = [
    "title",
    "complaint_title",
    "subject"
]

description_candidates = [
    "description",
    "complaint_description",
    "complaint_text",
    "text",
    "message"
]

target_candidates = [
    "category_title",
    "complaint_reason",
    "category",
    "reason"
]

title_column = next(
    (
        column
        for column in title_candidates
        if column in complaints_df.columns
    ),
    None
)

description_column = next(
    (
        column
        for column in description_candidates
        if column in complaints_df.columns
    ),
    None
)

target_column = next(
    (
        column
        for column in target_candidates
        if column in complaints_df.columns
    ),
    None
)

if description_column is None:
    raise ValueError(
        "No complaint-description column was found. "
        f"Available columns: {complaints_df.columns.tolist()}"
    )

if target_column is None:
    raise ValueError(
        "No complaint-reason column was found. "
        f"Available columns: {complaints_df.columns.tolist()}"
    )

if title_column is not None:
    complaints_df[title_column] = (
        complaints_df[title_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )
else:
    complaints_df["_temporary_title"] = ""
    title_column = "_temporary_title"

complaints_df[description_column] = (
    complaints_df[description_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

complaints_df[target_column] = (
    complaints_df[target_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

complaints_df["complaint_text"] = (
    complaints_df[title_column]
    + " "
    + complaints_df[description_column]
)

complaints_df["complaint_text"] = (
    complaints_df["complaint_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

def clean_complaint_text(value):
    value = str(value).lower()
    value = re.sub(r"http\S+|www\S+", " ", value)
    value = re.sub(r"\S+@\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"#(\w+)", r"\1", value)
    value = re.sub(r"&[a-z]+;", " ", value)
    value = re.sub(r"[^a-z\s]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

complaints_df["clean_text"] = (
    complaints_df["complaint_text"]
    .apply(clean_complaint_text)
)

complaints_df["complaint_reason"] = (
    complaints_df[target_column]
    .astype(str)
    .str.strip()
)

model_df = complaints_df[
    [
        "complaint_text",
        "clean_text",
        "complaint_reason"
    ]
].copy()

model_df = model_df.dropna(
    subset=[
        "clean_text",
        "complaint_reason"
    ]
)

invalid_labels = [
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "not available",
    "unknown",
    "-"
]

model_df = model_df[
    model_df["clean_text"].str.len() >= 5
].copy()

model_df = model_df[
    ~model_df["complaint_reason"]
    .str.lower()
    .isin(invalid_labels)
].copy()

model_df = model_df.drop_duplicates(
    subset=[
        "clean_text",
        "complaint_reason"
    ]
).reset_index(drop=True)

reason_counts = (
    model_df["complaint_reason"]
    .value_counts()
)

minimum_records_per_reason = 10

valid_reasons = reason_counts[
    reason_counts >= minimum_records_per_reason
].index

model_df = model_df[
    model_df["complaint_reason"].isin(valid_reasons)
].reset_index(drop=True)

if len(model_df) < 20:
    raise ValueError(
        "Not enough usable records are available."
    )

if model_df["complaint_reason"].nunique() < 2:
    raise ValueError(
        "At least two complaint-reason categories are required."
    )

print("\nComplaint-reason distribution:")
display(
    model_df["complaint_reason"]
    .value_counts()
    .rename_axis("complaint_reason")
    .reset_index(name="record_count")
)

X = model_df["clean_text"]
y = model_df["complaint_reason"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

complaint_reason_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=12000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        )
    )
])

complaint_reason_model.fit(
    X_train,
    y_train
)

y_pred = complaint_reason_model.predict(X_test)

prediction_probabilities = (
    complaint_reason_model.predict_proba(X_test)
)

prediction_confidence = (
    prediction_probabilities.max(axis=1)
)

accuracy = accuracy_score(y_test, y_pred)

weighted_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("\nModel trained successfully.")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of categories:", y.nunique())
print("Accuracy:", round(accuracy * 100, 2), "%")
print(
    "Weighted precision:",
    round(weighted_precision * 100, 2),
    "%"
)
print(
    "Weighted recall:",
    round(weighted_recall * 100, 2),
    "%"
)
print(
    "Weighted F1-score:",
    round(weighted_f1 * 100, 2),
    "%"
)
print(
    "Macro F1-score:",
    round(macro_f1 * 100, 2),
    "%"
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0
    )
).transpose().reset_index()

report_df = report_df.rename(
    columns={"index": "class_or_metric"}
)

class_labels = sorted(
    set(y_test.unique()).union(set(y_pred))
)

matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=class_labels
)

confusion_matrix_df = pd.DataFrame(
    matrix,
    index=class_labels,
    columns=class_labels
)

if len(class_labels) <= 20:
    fig, ax = plt.subplots(
        figsize=(12, 12)
    )

    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=class_labels
    )

    display_matrix.plot(
        ax=ax,
        values_format="d",
        xticks_rotation=90
    )

    plt.title(
        "Complaint Reason Classification Confusion Matrix"
    )
    plt.tight_layout()
    plt.show()

prediction_results = pd.DataFrame({
    "complaint_text": X_test.values,
    "actual_complaint_reason": y_test.values,
    "predicted_complaint_reason": y_pred,
    "prediction_confidence": prediction_confidence
})

prediction_results[
    "prediction_confidence_percentage"
] = (
    prediction_results[
        "prediction_confidence"
    ] * 100
).round(2)

prediction_results["prediction_correct"] = (
    prediction_results["actual_complaint_reason"]
    == prediction_results["predicted_complaint_reason"]
)

prediction_results.insert(
    0,
    "prediction_id",
    range(1, len(prediction_results) + 1)
)

evaluation_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1_score",
        "macro_f1_score",
        "original_records",
        "usable_records",
        "training_records",
        "testing_records",
        "number_of_categories"
    ],
    "value": [
        float(accuracy),
        float(weighted_precision),
        float(weighted_recall),
        float(weighted_f1),
        float(macro_f1),
        int(len(complaints_df)),
        int(len(model_df)),
        int(len(X_train)),
        int(len(X_test)),
        int(y.nunique())
    ]
})

labelled_data = model_df.copy()

labelled_data.insert(
    0,
    "labelled_record_id",
    range(1, len(labelled_data) + 1)
)

distribution_df = (
    model_df["complaint_reason"]
    .value_counts()
    .rename_axis("complaint_reason")
    .reset_index(name="record_count")
)

confusion_sql_df = (
    confusion_matrix_df
    .reset_index()
    .rename(
        columns={"index": "actual_complaint_reason"}
    )
)

joblib.dump(
    complaint_reason_model,
    "03_complaint_reason_classification_model.pkl"
)

prediction_results.to_csv(
    "03_complaint_reason_prediction_results.csv",
    index=False
)

evaluation_df.to_csv(
    "03_complaint_reason_model_evaluation.csv",
    index=False
)

report_df.to_csv(
    "03_complaint_reason_classification_report.csv",
    index=False
)

labelled_data.to_csv(
    "03_complaint_reason_labelled_data.csv",
    index=False
)

prediction_results.to_sql(
    name="complaint_reason_prediction_results",
    con=database_engine,
    if_exists="replace",
    index=False,
    chunksize=500
)

evaluation_df.to_sql(
    name="complaint_reason_model_evaluation",
    con=database_engine,
    if_exists="replace",
    index=False
)

report_df.to_sql(
    name="complaint_reason_classification_report",
    con=database_engine,
    if_exists="replace",
    index=False,
    chunksize=500
)

labelled_data.to_sql(
    name="complaint_reason_labelled_data",
    con=database_engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

distribution_df.to_sql(
    name="complaint_reason_class_distribution",
    con=database_engine,
    if_exists="replace",
    index=False
)

confusion_sql_df.to_sql(
    name="complaint_reason_confusion_matrix",
    con=database_engine,
    if_exists="replace",
    index=False
)

def predict_complaint_reason(new_complaint):
    cleaned_complaint = clean_complaint_text(
        new_complaint
    )

    if len(cleaned_complaint) < 3:
        raise ValueError(
            "Enter a meaningful complaint."
        )

    predicted_reason = complaint_reason_model.predict(
        [cleaned_complaint]
    )[0]

    probabilities = complaint_reason_model.predict_proba(
        [cleaned_complaint]
    )[0]

    probability_df = pd.DataFrame({
        "complaint_reason": complaint_reason_model.classes_,
        "probability_percentage": (
            probabilities * 100
        ).round(2)
    }).sort_values(
        "probability_percentage",
        ascending=False
    ).reset_index(drop=True)

    print("\nComplaint:", new_complaint)
    print("Predicted reason:", predicted_reason)
    print(
        "Confidence:",
        probability_df.iloc[0][
            "probability_percentage"
        ],
        "%"
    )

    display(probability_df.head(5))

    return predicted_reason

predict_complaint_reason(
    "Garbage has not been collected from our street "
    "for the last five days."
)

predict_complaint_reason(
    "The main road has many large potholes and "
    "vehicles are getting damaged."
)

predict_complaint_reason(
    "Streetlights in our colony are not working."
)

verification_df = pd.DataFrame({
    "table_name": [
        "complaint_reason_prediction_results",
        "complaint_reason_model_evaluation",
        "complaint_reason_classification_report",
        "complaint_reason_labelled_data",
        "complaint_reason_class_distribution",
        "complaint_reason_confusion_matrix"
    ],
    "total_rows": [
        len(prediction_results),
        len(evaluation_df),
        len(report_df),
        len(labelled_data),
        len(distribution_df),
        len(confusion_sql_df)
    ]
})

print("\nMySQL tables created:")
display(verification_df)

print("\n03_Complaint_Reason_Classification completed successfully.")

Enter MySQL password:  ········


Connected database: government_analytics
Dataset loaded from MySQL.

Complaint-reason distribution:


,complaint_reason,record_count
0,"Mobility - Roads, Footpaths and Infrastructure",4937
1,Garbage and Unsanitary Practices,3767
2,Traffic and Road Safety,920
3,Animal Husbandry,837
4,Yellow Spot,825
5,Street lighting,743
6,Streetlights,655
7,Pollution,435
8,Others,337
9,Water Supply and Services,316



Model trained successfully.
Training records: 12271
Testing records: 3068
Number of categories: 28
Accuracy: 75.55 %
Weighted precision: 77.14 %
Weighted recall: 75.55 %
Weighted F1-score: 75.82 %
Macro F1-score: 48.59 %

Classification report:
                                                precision    recall  f1-score   support

                              Animal Husbandry       0.93      0.93      0.93       167
                                  Certificates       0.82      0.90      0.86        10
         Community Infrastructure and Services       0.33      0.42      0.37        26
                                      Covid 19       0.00      0.00      0.00         2
                              Crime and Safety       0.39      0.46      0.43        28
                  Electricity and Power Supply       0.76      0.76      0.76        45
                                   Fire Safety       0.00      0.00      0.00         2
              Garbage and Unsanitary Practices   

,complaint_reason,probability_percentage
0,Garbage and Unsanitary Practices,76.09
1,Street lighting,3.49
2,Yellow Spot,2.23
3,Streetlights,2.20
4,Others,1.67


...


Complaint: Streetlights in our colony are not working.
Predicted reason: Streetlights
Confidence: 46.55 %


,complaint_reason,probability_percentage
0,Streetlights,46.55
1,Street lighting,30.17
2,Others,1.91
3,Yellow Spot,1.47
4,Traffic and Road Safety,1.45



MySQL tables created:


,table_name,total_rows
0,complaint_reason_prediction_results,3068
1,complaint_reason_model_evaluation,10
2,complaint_reason_classification_report,31
3,complaint_reason_labelled_data,15339
4,complaint_reason_class_distribution,28
5,complaint_reason_confusion_matrix,28



03_Complaint_Reason_Classification completed successfully.
